### Northwind refund scenario

Northwind's wiki, `refund-policy.md`, says a refund over $200 needs a manager. It does not mention an enterprise exception.

In meeting `2026-03-12-refund-review`, Nandini says at 00:03:12 (192000 ms) that an enterprise ticket with code R-14 can be refunded up to $500 without a manager. Omar agrees, Alex asks who owns the follow-up, and Priya assigns it to Sam.

In [1]:
import json
from pathlib import Path

ROOT=Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
SCENARIO_DIR=ROOT/"data"/"scenario"

policy=(SCENARIO_DIR/"refund-policy.md").read_text(encoding="utf-8")
raw_meeting=json.loads((SCENARIO_DIR/"meeting.json").read_text(encoding="utf-8"))

policy

'# Refund policy\n\n## Standard limit\n\nA refund over $200 needs a manager\n\n'

In [2]:
assert "200" in policy
assert "manager" in policy

Nandini=next(turn for turn in raw_meeting["turns"] if "R-14" in turn["text"])
assert Nandini["speaker"] == "Nandini"
assert Nandini["start_ms"] == 192000
assert raw_meeting["meeting_id"] == "2026-03-12-refund-review"
assert len(raw_meeting["turns"]) == 4

Nandini

{'turn_id': 't1',
 'speaker': 'Nandini',
 'start_ms': 192000,
 'end_ms': 208000,
 'text': 'An enterprise ticket with code R-14 can be refunded up to $500 without a manager'}

### Records

A turn is the spoken sentence. A record is a decision or an action drawn from one or more turns. `turn_ids` must name turns that exist. The owner can be someone other than the speaker: Sam owns the follow-up, and Nandini said so on t4.

In [3]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, model_validator

class Turn(BaseModel):
    turn_id: str
    speaker: str
    start_ms: int
    end_ms: int
    text: str

class Record(BaseModel):
    kind: Literal["decision", "problem", "action"]
    text: str
    owner: str
    turn_ids: list[str]=Field(min_length=1)

class Meeting(BaseModel):
    meeting_id: str
    turns: list[Turn]
    records: list[Record]=Field(default_factory=list)

    @model_validator(mode="after")
    def records_point_at_real_turns(self):
        known={turn.turn_id for turn in self.turns}
        for record in self.records:
            unknown=[turn_id for turn_id in record.turn_ids if turn_id not in known]
            if unknown:
                raise ValueError(f"Unknown turn_ids: {unknown}")
        return self

class Document(BaseModel):
    doc_id: str
    source_type: str
    title: str
    text: str
    section_path: str



In [4]:
meeting=Meeting.model_validate(raw_meeting)
wiki=Document(
    doc_id="refund-policy",
    source_type="wiki",
    title="Refund policy",
    text="A refund over $200 needs a manager",
    section_path="Standard limit",
)

meeting.records


[Record(kind='decision', text='An enterprise ticket with code R-14 can be refunded up to $500 without a manager', owner='Nandini', turn_ids=['t1']),
 Record(kind='action', text='Sam is the owner of the follow-up', owner='Sam', turn_ids=['t4'])]

In [5]:
bad= raw_meeting | {
    "records": [
        {
            "kind":"action",
            "text": "Sam is the owner of the follow-up",
            "owner": "Sam",
            "turn_ids": ["t9"]
        }
    ]
}

try:
    Meeting.model_validate(bad)
    raise AssertionError("t9 should have been rejected")
except ValidationError as e:
    print(e)


1 validation error for Meeting
  Value error, Unknown turn_ids: ['t9'] [type=value_error, input_value={'meeting_id': '2026-03-1...', 'turn_ids': ['t9']}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error


### Chunks

A chunk is the piece search will rank. A turn chunk keeps the speaker and the timestamp on the same sentence. A topic chunk keeps a few turns together for context. A record chunk is a decision or action and still points at the turns it came from. A section chunk is one wiki heading.

In [6]:
def format_ms(ms: int) -> str:
    total_seconds=ms // 1000
    hours=total_seconds // 3600
    minutes=(total_seconds % 3600) // 60
    seconds=total_seconds % 60
    return f"{hours}:{minutes:02d}:{seconds:02d}"

class Chunk(BaseModel):
    chunk_id: str
    text: str
    source_type: str
    unit: Literal["turn", "topic", "record", "section"]
    citation: dict



In [7]:
def chunk_meeting(meeting: Meeting)->list[Chunk]:
    turn_chunks=[]
    for turn in meeting.turns:
        stamp=format_ms(turn.start_ms)
        turn_chunks.append(
            Chunk(
                chunk_id=f"turn:{meeting.meeting_id}:{turn.turn_id}",
                text=f"{turn.speaker} ({stamp}):{turn.text}",
                source_type="meeting",
                unit="turn",
                citation={
                    "meeting_id": meeting.meeting_id,
                    "turn_id":turn.turn_id,
                    "speaker":turn.speaker,
                    "start_ms": turn.start_ms,
                    "end_ms":turn.end_ms,
                    "timestamp": stamp,
                },
            )
        )
    
    topic_chunks=[]
    window=3
    for start in range(0, len(turn_chunks)-window+1):
        group=turn_chunks[start: start+window]
        first=group[0].citation["turn_id"]
        last=group[-1].citation["turn_id"]
        topic_chunks.append(
            Chunk(
                chunk_id=f"topic:{meeting.meeting_id}:{first}-{last}",
                text="\n".join(chunk.text for chunk in group),
                source_type="meeting",
                unit="topic",
                citation={
                    "meeting_id": meeting.meeting_id,
                    "turns_ids": [chunk.citation["turn_id"] for chunk in group],
                },
            )
        )
    
    record_chunks=[]
    for record in meeting.records:
        joined="-".join(record.turn_ids)
        record_chunks.append(
            Chunk(
                chunk_id=f"record:{meeting.meeting_id}: {record.kind}:{joined}",
                text=f"{record.kind} by {record.owner}: {record.text}",
                source_type="meeting",
                unit="record",
                citation={
                    "meeting_id": meeting.meeting_id,
                    "kind":record.kind, 
                    "owner":record.owner,
                    "turn_ids": record.turn_ids,
                }
            )
        )
    
    return turn_chunks+topic_chunks+record_chunks

def chunk_document(document: Document)->Chunk:
    slug=document.section_path.lower().replace(" ","-")
    return Chunk(
        chunk_id=f"section:{document.doc_id}:{slug}",
        text=f"{document.section_path}: {document.text}",
        source_type=document.source_type,
        unit="section",
        citation={
            "doc_id": document.doc_id,
            "section_path": document.section_path,
        }
    )


In [8]:
chunks = chunk_meeting(meeting) + [chunk_document(wiki)]

r14 = next(chunk for chunk in chunks if chunk.unit == "turn" and "R-14" in chunk.text)
assert "Nandini" in r14.text
assert r14.citation["speaker"] == "Nandini"
assert r14.citation["start_ms"] == 192000

policy_chunk = next(chunk for chunk in chunks if chunk.unit == "section")
assert "200" in policy_chunk.text
assert policy_chunk.citation["section_path"] == "Standard limit"

assert {chunk.unit for chunk in chunks} == {"turn", "topic", "record", "section"}
# r14
policy_chunk

Chunk(chunk_id='section:refund-policy:standard-limit', text='Standard limit: A refund over $200 needs a manager', source_type='wiki', unit='section', citation={'doc_id': 'refund-policy', 'section_path': 'Standard limit'})

### Keyword search

BM25 ranks chunks by shared words. `R-14` has to match as a whole token, not as the separate letters R and 14. A score of 0 means the query and the chunk share nothing, so `Z-99` should return no hits.

In [10]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text: str)->list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

class BM25Index:
    def __init__(self,chunks:list[Chunk]):
        self.chunks=[chunk for chunk in chunks if chunk.unit != "topic"]
        corpus=[tokenize(chunk.text) for chunk in self.chunks]
        self.engine=BM25Okapi(corpus)

    def search(self, query:str, k:int=5)->list[tuple[Chunk, float]]:
        tokens=tokenize(query)
        if not tokens:
            return []
        scores=self.engine.get_scores(tokens)
        ranked=sorted(
            zip(self.chunks,scores),
            key=lambda pair: pair[1],
            reverse=True,
        )
        return [
            (chunk, float(score))
            for chunk,score in ranked[:k]
            if score>0
        ]


In [11]:
index=BM25Index(chunks)
hits=index.search("enterprise R-14 refund owner")

for chunk, score in hits:
    print(f"{score:.3f} {chunk.unit:8} {chunk.chunk_id}")

hit_ids={chunk.chunk_id for chunk,_ in hits}
assert any(chunk_id.startswith("turn:") and "t1" in chunk_id for chunk_id in hit_ids)
assert any(chunk.unit == "record" and "action" in chunk.chunk_id for chunk, _ in hits)
missing = index.search("Z-99")
assert missing == []
missing

1.645 section  section:refund-policy:standard-limit
1.279 record   record:2026-03-12-refund-review: decision:t1
1.241 turn     turn:2026-03-12-refund-review:t1
0.848 record   record:2026-03-12-refund-review: action:t4
0.815 turn     turn:2026-03-12-refund-review:t4


[]